# Task 4 model refresh
Copy the fitted winner because training images and labels are byte identical; link it to the new EDA card.

In [1]:
from pathlib import Path
import sys, json, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
SEED = set_seed()
old_card = json.loads((SRC/'task4/save/t4_lgbm_31_tuned_features_r1.json').read_text(encoding='utf8'))
new_card = json.loads((SRC/'task4/data/eda/eda_1_v2.json').read_text(encoding='utf8'))
old_data_card = json.loads((SRC/old_card['data_card_path']).read_text(encoding='utf8'))
old_train = pd.read_parquet(SRC/old_data_card['splits']['train']['path'])
new_train = pd.read_parquet(SRC/new_card['splits']['train']['path'])
assert old_train.image_sha256.equals(new_train.image_sha256)
assert old_train['class'].equals(new_train['class'])
exclude={'id','class','image_sha256','width','height'}
feature_cols=[c for c in new_train if c not in exclude]
assert feature_cols == [c for c in old_train if c not in exclude]
y = new_train['class']
groups = new_train.image_sha256
splitter = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
dev_idx,valid_idx=next(splitter.split(new_train,y,groups))
assert not (set(groups.iloc[dev_idx]) & set(groups.iloc[valid_idx]))
model=joblib.load(SRC/old_card['model_path'])
assert model.n_features_in_ == len(feature_cols)
X_valid=new_train.iloc[valid_idx][feature_cols].astype('float32')
pred=model.predict(X_valid)
score=f1_score(y.iloc[valid_idx],pred,average='macro')
old_eval=json.loads((SRC/'task4/save/t4_lgbm_31_tuned_features_r1-evaluation.json').read_text(encoding='utf8'))
assert abs(score-old_eval['score']) < 1e-12
print('Validation macro F1:',score)
name='t4_lgbm_31_tuned_features_r2'
if not (SRC/'task4/save'/f'{name}.json').exists():
    save_model(task=4,name=name,model=model,data_name='eda_1_v2',card={
        'idea':'Exact copy of previously fitted LightGBM model; train images and labels unchanged',
        'source_model':old_card['name'],'seed':SEED,'validation':'same stratified image-hash fold 1/5',
        'test_change':'new manifest IDs/order; no test fit'})
    evaluate_model(task=4,name=name,y_true=y.iloc[valid_idx],y_pred=pred,
        card={'split':'same stratified image-hash fold 1/5','source_model':old_card['name'],'seed':SEED})


Validation macro F1: 0.7716444859086071


In [2]:
# Copy every other fitted candidate; training images and labels are unchanged.
old_save = SRC / 'task4' / 'save'
old_cards = sorted(p for p in old_save.glob('*.json') if p.stem.endswith('_r1'))
assert len(old_cards) == 28
for path in old_cards:
    old = json.loads(path.read_text(encoding='utf8'))
    old_name = old['name']
    new_name = old_name.removesuffix('_r1') + '_r2'
    if new_name == name:
        continue
    data_name = {'eda_original_v1':'eda_original_v2', 'eda_1_v1':'eda_1_v2'}[old['data_name']]
    data_card = json.loads((SRC/'task4/data/eda'/f'{data_name}.json').read_text(encoding='utf8'))
    current_train = pd.read_parquet(SRC/data_card['splits']['train']['path'])
    old_data = json.loads((SRC/old['data_card_path']).read_text(encoding='utf8'))
    previous_train = pd.read_parquet(SRC/old_data['splits']['train']['path'])
    assert current_train.image_sha256.equals(previous_train.image_sha256)
    assert current_train['class'].equals(previous_train['class'])
    cols = [c for c in current_train if c not in exclude]
    assert cols == [c for c in previous_train if c not in exclude]
    if (old_save/f'{new_name}.json').exists():
        continue
    fitted = joblib.load(SRC/old['model_path'])
    predictions = fitted.predict(current_train.iloc[valid_idx][cols].astype('float32'))
    old_result = json.loads((old_save/f'{old_name}-evaluation.json').read_text(encoding='utf8'))
    current_score = f1_score(y.iloc[valid_idx], predictions, average='macro')
    assert abs(current_score-old_result['score']) < 1e-12, old_name
    save_model(task=4,name=new_name,model=fitted,data_name=data_name,
        card={'idea':'Exact copy of fitted candidate; train images and labels unchanged',
              'source_model':old_name,'seed':SEED,'test_change':'new manifest IDs/order; no test fit'})
    evaluate_model(task=4,name=new_name,y_true=y.iloc[valid_idx],y_pred=predictions,
        card={'split':'same stratified image-hash fold 1/5','source_model':old_name,'seed':SEED})
    print(new_name, round(current_score,5))
print('Refreshed models:',len(list(old_save.glob('*_r2.joblib'))))

t4_dummy_pixels_r2 0.11305
t4_dummy_tuned_pixels_r2 0.20404


t4_extratrees_features_r2 0.61219


t4_extratrees_pixels_r2 0.5257


t4_extratrees_tuned_features_r2 0.59999


t4_extratrees_tuned_pixels_r2 0.49213


t4_hgb_features_r2 0.71047


... (16 outputs left out)

t4_svm_features_r2 0.67859


t4_svm_pixels_r2 0.33276


t4_svm_tuned_features_r2 0.67655


t4_svm_tuned_pixels_r2 0.35987
Refreshed models: 28
